In [84]:
import pandas as pd

In [85]:
train = pd.read_csv("dataset/train_data.csv")
test = pd.read_csv("dataset/test_data.csv")

output = pd.DataFrame(columns=["subtaskID", "datapointID", "answer"])

In [86]:
count = (
    (test["Sex"] == "Femelă") &
    test["Ureche_lăsată"] &
    (test["Culoare"] == "Havana")
).sum()

print(count)

output.loc[0] = [1, 1, count]

11


In [87]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()
le = LabelEncoder()

st2 = train.drop(columns=["ID", "Sex", "Vârstă", "Sănătate", "Scor_jurizare"])
st2_test = test.drop(columns=["ID", "Sex", "Vârstă", "Sănătate"])
cols = ["Calitate_blană", "Culoare", "Tip_blană", "Formă_corp"]

st2 = pd.get_dummies(st2, columns=cols)
st2_test = pd.get_dummies(st2_test, columns=cols)

st2 = pd.DataFrame(
    scaler.fit_transform(st2),
    columns=st2.columns,
    index=st2.index
)

st2_test = pd.DataFrame(
    scaler.transform(st2_test),
    columns=st2_test.columns,
    index=st2_test.index
)

model = KMeans(n_clusters=3, random_state=42)

model.fit(st2)
preds = model.predict(st2_test)

output = pd.concat([output, pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test["ID"],
    "answer": preds
})])

st2.head()

,Greutate,Lungime_urechi,Ureche_lăsată,Apare_gușa,Calitate_blană_Bună,Calitate_blană_Excelentă,Calitate_blană_Foarte_bună,Culoare_Agouti,Culoare_Alb,Culoare_Albastru,Culoare_Havana,Culoare_Negru,Tip_blană_Lungă,Tip_blană_Scurtă,Formă_corp_Aproape_corectă,Formă_corp_Corectă,Formă_corp_Incorectă
0,-0.967240,-0.913744,-0.697280,-0.533295,-0.681495,1.452966,-0.752327,-0.539919,-0.517799,-0.484322,2.064742,-0.47305,-1.452966,1.452966,-0.628031,-0.750000,1.341641
1,-0.472570,-0.601536,-0.697280,-0.533295,-0.681495,1.452966,-0.752327,1.852128,-0.517799,-0.484322,-0.484322,-0.47305,0.688247,-0.688247,1.592279,-0.750000,-0.745356
2,1.775928,1.500666,1.434144,-0.533295,-0.681495,-0.688247,1.329210,-0.539919,1.931251,-0.484322,-0.484322,-0.47305,0.688247,-0.688247,-0.628031,1.333333,-0.745356
3,-0.877300,-0.840895,-0.697280,-0.533295,-0.681495,-0.688247,1.329210,-0.539919,-0.517799,-0.484322,2.064742,-0.47305,-1.452966,1.452966,-0.628031,-0.750000,1.341641
4,1.775928,1.511072,1.434144,1.875134,-0.681495,-0.688247,1.329210,-0.539919,-0.517799,-0.484322,2.064742,-0.47305,0.688247,-0.688247,-0.628031,-0.750000,1.341641


In [88]:
from sklearn.metrics import silhouette_score

score = silhouette_score(st2_test, preds)
print(score)

0.16961696193951895


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

cat_cols = [
    "Sex", "Ureche_lăsată", "Culoare", "Tip_blană",
    "Calitate_blană", "Formă_corp", "Apare_gușa", "Sănătate"
]

st3 = train.drop(columns=["ID"])
st3_test = test.drop(columns=["ID"])

st3 = pd.get_dummies(st3, columns=cat_cols)
st3_test = pd.get_dummies(st3_test, columns=cat_cols)

X = st3.drop(columns=["Scor_jurizare"])
y = st3["Scor_jurizare"]

# make submit test match train columns exactly
st3_test = st3_test.reindex(columns=X.columns, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

preds = model.predict(X_val)
print("MSE:", mean_squared_error(y_val, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_val, preds)))

test_preds = model.predict(st3_test)

submission = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": test["ID"].astype(int),
    "answer": test_preds
})

output = pd.concat([output, submission], ignore_index=True)

output["subtaskID"] = output["subtaskID"].astype(int)
output["datapointID"] = output["datapointID"].astype(int)

MSE: 0.9117668380952382
RMSE: 0.9548648271327403


In [90]:
output.to_csv("output.csv", index=False)